# Programmation Python  pour les mathématiques

**Julien Guillod**, [Sorbonne Université](http://www.sorbonne-universite.fr/),
Licence <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND</a>

L'entier des chapitres est disponible au format
[HTML](https://python.guillod.org//) et [PDF](https://python.guillod.org//python.pdf).
Ce notebook peut également être exécuté sur [GESIS](https://notebooks.gesis.org/binder/v2/gh/guillod/python/master?filepath=chap08.ipynb).

# 8 Théorie des graphes

<div id="ch:graphes"></div>

Un graphe est un couple $G = (X, E)$ constitué d'un ensemble $X$, non vide et fini, et d'un ensemble $E$ de paires d'éléments de $X$. Les éléments de $X$ sont les sommets du graphe $G$, ceux de $E$ sont les arêtes du graphe $G$.
Un graphe est orienté si les arêtes ont une direction, c'est-à-dire si les couples d'éléments de $E$ sont des listes ordonnées, c'est-à-dire telle que $(i,j) \in E$ n'est pas équivalent à $(j,i)\in E$. Ici seuls des graphes non-orientés, c'est-à-dire dont les paires d'éléments de $X$ sont des ensembles non ordonnés ($\{i,j\} \in E$) sont considérés.

Par exemple, le graphe complet à $n$ sommets $K_n$ est le graphe de sommets $X = {1,2,\dots,n}$ ayant comme arêtes les parties à deux éléments de $X$. En particulier, $K_4 = (X,E)$ où $X=\{1,2,3,4\}$ et $E = \big\{ \{1,2\}, \{1,3\}, \{1,4\}, \{2, 3\}, \{2, 4\}, \{3, 4\}  \big\}$.

**Concepts abordés:**

* graphes non-orientés

* représentation comme dictionnaire

* utilisation des frozensets

* matrice d'adjacence

* recherche de chemins et de triangles

* fonction récursive



# Exercice 8.1: Graphes comme dictionnaires

Une façon de représenter un graphe $G$ est de le faire avec un dictionnaire dont les clés sont
les sommets, et la valeur associée à chaque clé $x \in X$ est un ensemble contenant les voisins de $x$.

**a)**
Construire sous forme de dictionnaire les graphes suivants:

<center><img src="https://python.guillod.org/fig/graphes.png" style="width:90%;max-width:800px;"></center>



In [22]:
Ga = {1: {2,5}, 2:{1,3,5}, 3:{2,4}, 4:{3,5,6}, 5:{1,2,4}, 6:{4}}
Gb = {i: {(a+i) % 5 for a in {1,2,3,4}} for i in range(5)}
Gc = {}
# sommets externes
Gc1 = {i: {(a+i) % 5 for a in {1,4}}.union({i+5}) for i in range(5)}
Gc.update(Gc1)
# sommets internes
Gc2 = {i: {5 + (a+i) % 5 for a in {2,3}}.union({i-5}) for i in range(5,10)}
Gc.update(Gc2)


**b)**
Écrire une fonction `complet(n)` qui construit comme dictionnaire le graphe complet $K_n$.



In [23]:
def complet(n):
    all = set(range(0,n))
    return {i: all-{i} for i in range(n)}  


**c)**
Un graphe donné sous forme de dictionnaire contient l'information plusieurs fois.  Écrire une fonction `corriger(graphe)` permettant de rajouter les éléments manquants de sorte que pour tout sommet `x`, si `y` appartient à `graphe[x]`, alors `y` est aussi une clé et `x` appartient à `graphe[y]`. Tester cette fonction.



In [24]:
def corriger(graph):
    # graphe de sortie
    out = graph.copy()
    # boucles sur toutes les arêtes
    for s in graph.keys():
        for a in graph[s]:
            # si la clef correspondante n'existe pas
            if a not in out.keys():
                # rajoute la clef et la valeur associée
                out[a] = {s}
            # si la clef correspondante existe
            else:
                # rajoute la valeur
                out[a].add(s)
    return out
corriger({1:{3,4,2},3:{2}})

{1: {2, 3, 4}, 3: {1, 2}, 2: {1, 3}, 4: {1}}


**d)**
Écrire une fonction qui retourne l'ensemble (type `set`) de toutes les arêtes d'un graphe représenté par un dictionnaire.

**Indication:**
Les ensembles sont mutables et donc pas hashables.



In [25]:
def aretes(graph):
    # ensemble des arêtes
    out = set()
    # boucles sur toutes les arêtes
    for s in graph.keys():
        for a in graph[s]:
            out.add(frozenset((s,a)))
    return out


**e)**
<span style="color:red">!</span> Écrire une fonction permettant de déterminer si deux sommets sont connectés par un chemin ou non et retourne le chemin si oui.

**Indication:**
Utiliser une fonction récursive.



In [26]:
def find_path(graph, start, end, path=[]):
    # rajoute au chemin sans modifier l'argument
    path = path + [start]
    # termine si le chemin a abouti
    if start == end:
        return path
    # termine si start est isolé
    if start not in graph.keys():
        return None
    # boucle sur les sommets adjacents pas encore dans le chemin
    for node in graph[start]:
        if node not in path:
            # recherche un nouveau chemin depuis ici
            newpath = find_path(graph, node, end, path)
            # s'arrête si le nouveau chemin est OK
            if newpath: return newpath
    return None
find_path(Ga,1,6)
find_path(Gc,0,7)

[0, 1, 2, 3, 8, 5, 7]


**f)**
<span style="color:red">!</span> Écrire une fonction qui retourne tous les chemins entre deux sommets (sans les cycles).



In [27]:
def find_paths(graph, start, end, path=[]):
    # rajoute au chemin sans modifier l'argument
    path = path + [start]
    # termine si le chemin a abouti
    if start == end:
        return [path]
    # termine si start est isolé
    if start not in graph.keys():
        return []
    # liste des chemins
    paths = []
    # boucle sur les sommets adjacents pas encore dans le chemin
    for node in graph[start]:
        if node not in path:
            # recherche les nouveaux chemins depuis ici
            newpaths = find_paths(graph, node, end, path)
            paths += newpaths
    return paths

# Exercice 8.2: Triangles dans un graphe

Un triangle dans un graphe est un ensemble de trois sommets reliés entre eux par trois arêtes. La recherche et l'analyse des triangles dans un graphe est importante pour comprendre sa structure.

**a)**
Déterminer mathématiquement le nombre de sous-ensembles de cardinal trois que possède un ensemble de sommets $X$.




**b)**
Écrire une fonction retournant l'ensemble des triangles d'un graphe.



In [28]:
def triangles(graph):
    # ensemble des triangles
    out = set()
    # boucle sur les arêtes
    for i in graph.keys():
        for j in graph[i]:
            # itère sur la fleur de i
            for k in graph[i]:
                # si c'est un triangle
                if k in graph[j]:
                    out.add(frozenset((i,j,k)))
    return out


A chaque graphe $G=(X,E)$ correspond une unique matrice $A$ symétrique de taille $n \times n$ avec $n=|X|$ définie par:

$$
A_{ij}=\begin{cases}
1\,, & \text{si}\;\{i,j\}\in E\,,\\ 
0\,, & \text{si}\;\{i,j\}\notin E\,.
\end{cases}
$$

Cette matrice est appelée la matrice d'adjacence du graphe $G$.

**c)**
Définir une fonction retournant la matrice d'adjacence d'un graphe.

**Indication:**
Faire attention que les sommets ne sont pas forcément indexés par des entiers compris entre $0$ et $n$ dans le dictionnaire.



In [29]:
import numpy as np
def adjacence(graph):
    # dimension du graphe
    n = len(graph)
    # matrice d'adjacence
    A = np.zeros((n,n), dtype=int)
    # définit un dictionnaire de correspondance sommet -> index entre 0 et n
    dico = {i: idx for idx,i in enumerate(graph.keys())}
    for i in graph.keys():
        for j in graph[i]:
            A[dico[i],dico[j]] = 1
    return A


**d)**
Définir une fonction ayant pour argument une matrice d'adjacence et retournant le graphe correspondant sous forme d'un dictionnaire.



In [30]:
def graphe(A): # on suppose la matrice carrée
    n = len(A)
    return {i: {j for j in range(n) if A[i][j]==1} for i in range(n)}

In [31]:
def graphe(A):
    graph = dict()
    for i,line in enumerate(A):
        # indices des 1 présents sur cette ligne
        indices, = np.where(line)
        # ajoute la clef i et les indices correspondants
        graph[i] = set(indices)
    return graph


**e)**
En utilisant la matrice d'adjacence $A$ et la matrice $B=A^2$, écrire une fonction retournant l'ensemble des triangles d'un graphe.



In [20]:
def triangles2(A):
    # ensemble de triangles
    out = set()
    # produit matriciel
    B = A @ A
    # matrice ayant True si A_ij \neq 0 et B_ij \neq 0
    cond = (A !=0) & (B !=0)
    # indices des arêtes à partir desquels un triangle existe
    liste_i,liste_j = np.where(cond)
    for i,j in zip(liste_i,liste_j):
        # liste des indices k tels que Aik=1 et A_jk=1
        indices, = np.where( (A[i]==1) & (A[j]==1) )
        # tous ces indices sont des triangles
        for k in indices:
            out.add(frozenset((i,j,k)))
    return out

In [43]:
# matrice aléatoire de 0 et 1
A=np.random.binomial(1,0.5,(4,4))
print(A)
# symétrise la matrice
print(A.T)
A = A & A.T
print(A)
# test
triangles2(A) == triangles(graphe(A))

[[0 1 0 1]
 [1 0 1 0]
 [0 1 1 1]
 [1 0 0 0]]
[[0 1 0 1]
 [1 0 1 0]
 [0 1 1 0]
 [1 0 1 0]]
[[0 1 0 1]
 [1 0 1 0]
 [0 1 1 0]
 [1 0 0 0]]


True


**f)**
En utilisant la matrice d'adjacence $A$, écrire une fonction calculant le nombre de triangles.

**Indication:**
Interpréter les entrées de la matrice $A^n$.



Solution: Le nombre de chemins distincts de longueur 
n
 entre 
i
 et 
j
 est donné par 
C
i
j
 où 
C
=
A
n
. Ainsi la trace de 
A
3
 représente le nombre de triangles présents dans le graphe avec leur multiplicité. Vu que les triangles peuvent commencer avec un des trois sommets différents et dans deux directions différentes, le nombre de triangles distincts est 
tr
(
A
3
)
6
:

In [ ]:
def nombre_triangles(A):
    # calcule A^3 au sens matriciel
    B = np.linalg.matrix_power(A,3)
    return np.trace(B)//6

# Exercice 8.3: <span style="color:red">!!</span> Module NetworkX

De nombreux algorithmes de la théorie des graphes sont implémentés dans le module networkX, voir la documentation [ici](https://networkx.github.io/documentation/).

**a)**
Suivre le tutoriel de networkX disponible [ici](https://networkx.github.io/documentation/stable/tutorial.html).

**b)**
Analyser un des graphes téléchargeables à l'adresse: <https://github.com/gephi/gephi/wiki/Datasets> ou <https://snap.stanford.edu/data/>.

